### 1. Environment Setup & Library Imports

In [1]:
import pandas as pd
from e_commerce_sales_analysis.db import load_table, get_engine
from e_commerce_sales_analysis.cleaning import clean_city

python-dotenv could not parse statement starting at line 16


### Project Scope: Brazilian E-Commerce (Olist) Sales & Logistics Analysis
- **Extract**: Ingest relational e-commerce tables from MySQL using connection pooling and chunked loaders.
- **Clean & Validate**: Standardize geographic names (Portuguese diacritics/accents), resolve foreign key anomalies, and preserve valid business entities.
- **Model**: Construct a Star Schema data model (Fact and Dimension tables) to prevent row explosion and revenue double-counting.
- **Export**: Store production-ready datasets in columnar Parquet format for SQL analytics and Power BI reporting.

### Text Standardization Function: `clean_city`
Brazilian address data contains multiple inconsistencies:
- Special accented Portuguese characters (e.g., *São Paulo*, *Brasília*, *Poços de Caldas*)
- Trailing two-letter state abbreviations (e.g., *Rio de Janeiro - RJ*, *Curitiba/PR*)
- Punctuation and hyphens (*Belo-Horizonte*)
- Inconsistent casing and trailing whitespace

The `clean_city` function in `e_commerce_sales_analysis.cleaning.text` standardizes these to clean ASCII title-case strings.

### Dataset Inspection: `olist_customers`
**Schema Definition:**
- `customer_id`: Unique transaction-level identifier generated per purchase.
- `customer_unique_id`: Unique identifier for the actual individual customer (persistent across repeat orders).
- `customer_zip_code_prefix`: 5-digit postal code prefix.
- `customer_city`: City name.
- `customer_state`: 2-letter state code.

**Key Observations:**
- Contains 99,441 rows with zero missing values.
- `customer_id` has 99,441 unique values, while `customer_unique_id` has 96,096 unique values.
- The difference (3,345 records) represents repeat customer orders. We must retain `customer_unique_id` throughout the pipeline to measure repeat purchase rates and Customer Lifetime Value (LTV).

In [2]:
customer = load_table("olist_customers", chunk_size=50_000)

In [3]:
customer_copy = customer.copy()

In [4]:
customer_copy.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,00012a2ce6f8dcda20d059ce98491703,248ffe10d632bebe4f7267f1f44844c9,06273,osasco,SP
1,000161a058600d5901f007fab4c27140,b0015e09bb4b6e47c52844fab5fb6638,35550,itapecerica,MG
2,0001fd6190edaaf884bcaf3d49edf079,94b11d37cd61cb2994a194d11f89682b,29830,nova venecia,ES
3,0002414f95344307404f0ace7a26f1d5,4893ad4ea28b2c5b3ddf4e82e79db9e6,39664,mendonca,MG
4,000379cdec625522490c315e70c7a9fb,0b83f73b19c2019e182fd552c048a22c,04841,sao paulo,SP


In [5]:
customer_copy.tail()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
99436,fffecc9f79fd8c764f843e9951b11341,e5794df8573fa179a90a7b797fc4b71f,95630,parobe,RS
99437,fffeda5b6d849fbd39689bb92087f431,afbb5a642107cf6bb1ca68e863175f03,22461,rio de janeiro,RJ
99438,ffff42319e9b2d713724ae527742af25,680213db6ebd9e4f24d03280cbe10346,06754,taboao da serra,SP
99439,ffffa3172527f765de70084a7e53aae8,48fd7dec70f2b104a1d5e8c5c639102b,37130,alfenas,MG
99440,ffffe8b65bbe3087b653a978c870db99,736e6bfa0510aa5b878881a226a5fd89,06172,osasco,SP


In [6]:
customer_copy.sample(5)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
11056,1c5c0870a06762e283d8939e79241a4c,f6aaac5f5b7ab52ab49bdd22d0225a0a,14810,araraquara,SP
28447,495b67cd5a866ee9a61ea6e8b4cadb51,2e43e031f10de28e557c35ef668f9396,92410,canoas,RS
74267,be7fd77bf39879f67d00f9f8c433b46d,f6b343f83417e0da74b76fa002629c91,35500,divinopolis,MG
62198,a01db3d31374817c4f5694fa39498580,fd26a250637b67b1ab80793a71e33224,03264,sao paulo,SP
4894,0ca5257ba1280eb54d0e7c5823f8f9b4,e5f48bc167ddec4bc538a97e9d689ddb,02324,sao paulo,SP


In [7]:
customer_copy.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  str  
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: str(5)
memory usage: 11.5 MB


In [8]:
customer_copy.shape

(99441, 5)

In [9]:
customer_copy["customer_city"] = customer_copy["customer_city"].apply(
    lambda x: x.strip().title() if isinstance(x, str) else x
)

In [10]:
print(customer_copy['customer_id'].nunique())

99441


In [11]:
print(customer_copy['customer_unique_id'].nunique())

96096


In [12]:
print(99441 - 96096)

3345


In [13]:
sum(customer_copy['customer_id'] == customer_copy['customer_unique_id'])

0

In [14]:
customer_copy['customer_id'].duplicated().sum()

np.int64(0)

In [15]:
customer_copy.duplicated().sum()

np.int64(0)

In [16]:
customer_copy.isnull().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [17]:
# Verify cleaned customer city distribution
print("Unique customer cities:", customer_copy['customer_city'].nunique())
customer_copy[['customer_city', 'customer_state']].head()

Unique customer cities: 4119


,customer_city,customer_state
0,Osasco,SP
1,Itapecerica,MG
2,Nova Venecia,ES
3,Mendonca,MG
4,Sao Paulo,SP


### Dataset Inspection: `olist_geolocation`
**Schema Definition:**
- `geolocation_zip_code_prefix`: 5-digit postal code prefix.
- `geolocation_lat` & `geolocation_lng`: Geographic coordinates.
- `geolocation_city`: City name.
- `geolocation_state`: 2-letter state code.

**Key Observations:**
- Contains 1,000,163 records representing geographic coordinate samples across Brazil.
- Raw city names contain accented characters, punctuation, and state abbreviations.
- Applying `clean_city` reduces unique city representations from 8,011 down to 5,947, successfully consolidating misspelled and accented variations.
- A small dictionary of explicit city name overrides is applied to align legacy municipal names (e.g., *Embu* -> *Embu Das Artes*, *Parati* -> *Paraty*) for map visualizations.

In [18]:
geo = load_table("olist_geolocation", chunk_size=50_000)

In [19]:
geo_copy = geo.copy()

In [20]:
geo_copy.head()

,geolocation_id,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1,01037,-23.545621,-46.639292,sao paulo,SP
1,2,01046,-23.546081,-46.644820,sao paulo,SP
2,3,01046,-23.546129,-46.642951,sao paulo,SP
3,4,01041,-23.544392,-46.639499,sao paulo,SP
4,5,01035,-23.541578,-46.641607,sao paulo,SP


In [21]:
geo_copy.tail()

,geolocation_id,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1000158,1000159,99950,-28.068639,-52.010705,tapejara,RS
1000159,1000160,99900,-27.877125,-52.224882,getulio vargas,RS
1000160,1000161,99950,-28.071855,-52.014716,tapejara,RS
1000161,1000162,99980,-28.388932,-51.846871,david canabarro,RS
1000162,1000163,99950,-28.070104,-52.018658,tapejara,RS


In [22]:
geo_copy.sample(5)

,geolocation_id,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
971527,971528,95670,-29.359800,-50.854016,gramado,RS
433579,433580,21765,-22.892832,-43.430464,rio de janeiro,RJ
828110,828111,78652,-10.651447,-51.571254,confresa,MT
144959,144960,05854,-23.662058,-46.761643,sao paulo,SP
260003,260004,11677,-23.584460,-45.337197,caraguatatuba,SP


In [23]:
geo_copy.shape

(1000163, 6)

In [24]:
geo_copy.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 6 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_id               1000163 non-null  int64  
 1   geolocation_zip_code_prefix  1000163 non-null  str    
 2   geolocation_lat              1000163 non-null  float64
 3   geolocation_lng              1000163 non-null  float64
 4   geolocation_city             1000163 non-null  str    
 5   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(3)
memory usage: 62.5 MB


In [25]:
geo_copy.isna().sum()

geolocation_id                 0
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [26]:
geo_copy['geolocation_city'].value_counts()

geolocation_city
sao paulo                     135800
rio de janeiro                 62151
belo horizonte                 27805
são paulo                      24918
curitiba                       16593
                               ...  
são josé do herval                 1
almirante tamandaré do sul         1
são valentim                       1
três arroios                       1
ciríaco                            1
Name: count, Length: 8011, dtype: int64

In [27]:
print(geo_copy['geolocation_city'].nunique())

8011


In [28]:
geo_copy["geolocation_city"] = clean_city(geo_copy["geolocation_city"])


In [29]:
print(geo_copy['geolocation_city'].nunique())

5947


In [30]:
geo_copy['geolocation_city'].str.contains(r'\d+|\W+').sum()

np.int64(536112)

In [31]:
print(min(geo_copy['geolocation_lat']), max(geo_copy['geolocation_lat']))
print(min(geo_copy['geolocation_lng']), max(geo_copy['geolocation_lng']))

-36.6053744107061 45.06593318269697
-101.46676644931476 121.10539381057764


In [32]:
# Verify geolocation city counts before applying manual overrides
print("Unique geolocation cities before overrides:", geo_copy['geolocation_city'].nunique())

Unique geolocation cities before overrides: 5947


In [33]:
city_overrides = {
    "Rio De Janeiro Rio De Janeiro": "Rio De Janeiro",
    "Embu": "Embu Das Artes",
    "Santa Barbara Doeste": "Santa Barbara D Oeste",
    "Sao Joao Do Pau Dalho": "Sao Joao Do Pau D Alho",
    "Parati": "Paraty",
    }
geo_copy["geolocation_city"] = geo_copy["geolocation_city"].replace(city_overrides)

In [34]:
geo_copy.duplicated().sum()

np.int64(0)

In [35]:
geo_copy['geolocation_city'].nunique()

5943

### Dataset Inspection: `olist_products` & Category Translations
**Schema Definition:**
- `product_id`: Unique product hash.
- `product_category_name`: Portuguese category name.
- Product metrics: photo count, weight (grams), and dimensional sizes (length, height, width in cm).

**Key Observations:**
- Categories are originally in Portuguese; we map them to English using `product_category_name_translation`.
- Two categories (`pc_gamer` and `portateis_casa_forno_e_cafe`) were missing in the Kaggle translation file; we explicitly map them to ensure complete category coverage.
- Missing values in `product_category_name` are labeled as `'None'` to prevent silent drops during joins.
- Missing photo counts are imputed using the median value (1.0).
- We preserve physical dimensions and `product_weight_g` to enable freight cost and shipping fee analytics.

In [36]:
products = load_table("olist_products", chunk_size=50_000)

In [37]:
products_copy = products.copy()

In [38]:
product_translation = load_table("product_category_name_translation", chunk_size=50_000)

In [39]:
product_translation_copy = product_translation.copy()

In [40]:
products_copy.shape

(32951, 9)

In [41]:
product_translation_copy.shape

(73, 2)

In [42]:
product_translation_copy.head()

,product_category_name,product_category_name_english
0,agro_industria_e_comercio,agro_industry_and_commerce
1,alimentos,food
2,alimentos_bebidas,food_drink
3,artes,art
4,artes_e_artesanato,arts_and_craftmanship


In [43]:
products_copy['product_category_name'].nunique()

73

In [44]:
product_translation_copy['product_category_name_english'].nunique()

73

In [45]:
missing_cats = products_copy[
    ~products_copy["product_category_name"].isin(
        product_translation_copy["product_category_name"]
    )
]["product_category_name"].unique().tolist()
print(missing_cats)

[nan]


In [46]:
sorted_cats = products_copy.sort_values(
    by=["product_category_name"], ascending=False
)["product_category_name"].unique().tolist()

In [47]:
product_translation_copy[product_translation_copy['product_category_name'] == 'pcs']

,product_category_name,product_category_name_english
61,pcs,computers


In [48]:
product_translation_copy[
    product_translation_copy["product_category_name"]
    == "portateis_casa_forno_e_cafe"
]

,product_category_name,product_category_name_english
64,portateis_casa_forno_e_cafe,small_appliances_home_oven_and_coffee


In [49]:
print(product_translation_copy.columns)
print(products_copy.columns)

Index(['product_category_name', 'product_category_name_english'], dtype='str')
Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='str')


In [50]:
products_copy = pd.merge(
    products_copy,
    product_translation_copy,
    how="left",
    on="product_category_name",
)

In [51]:
products_copy.loc[
    products_copy["product_category_name"] == "pc_gamer",
    "product_category_name_english",
] = "Computers"
products_copy.loc[
    products_copy["product_category_name"] == "portateis_casa_forno_e_cafe",
    "product_category_name_english",
] = "small_appliances_home_oven_and_coffee"

In [52]:
products_copy[products_copy['product_category_name'] == 'pc_gamer']

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
120,0105b5323d24fc655f73052694dbbb3a,pc_gamer,59.0,621.0,4.0,2839.0,19.0,16.0,18.0,Computers
13329,6727051471a0fc4a0e7737b57bff2549,pc_gamer,60.0,1532.0,3.0,650.0,16.0,22.0,20.0,Computers
28348,dbe520fb381ad695a7e1f2807d20c765,pc_gamer,60.0,840.0,6.0,800.0,18.0,22.0,22.0,Computers


In [53]:
products_copy[products_copy['product_category_name'] == 'portateis_casa_forno_e_cafe']

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
503,0433830caca22b01a0f477d31307b043,portateis_casa_forno_e_cafe,47.0,2810.0,7.0,7450.0,40.0,10.0,40.0,small_appliances_home_oven_and_coffee
3146,17ec1cf6dc8dc7557ca34f1d64a93dc7,portateis_casa_forno_e_cafe,45.0,537.0,1.0,1600.0,18.0,30.0,24.0,small_appliances_home_oven_and_coffee
3986,1dff983bf86722fbc11a0c1d0ec13624,portateis_casa_forno_e_cafe,60.0,1179.0,1.0,400.0,28.0,11.0,11.0,small_appliances_home_oven_and_coffee
4310,206ee9f28d05f04f891e3cef4b220dd1,portateis_casa_forno_e_cafe,51.0,688.0,1.0,1200.0,25.0,25.0,25.0,small_appliances_home_oven_and_coffee
4554,224037b21bfa8621e39c265513ee6594,portateis_casa_forno_e_cafe,50.0,345.0,1.0,16550.0,40.0,60.0,40.0,small_appliances_home_oven_and_coffee
5182,27274145782d0048827cb28b7786323b,portateis_casa_forno_e_cafe,50.0,2626.0,6.0,5217.0,48.0,36.0,16.0,small_appliances_home_oven_and_coffee
6399,31240eaae01b43247d878f3e89996e35,portateis_casa_forno_e_cafe,39.0,345.0,1.0,400.0,16.0,17.0,20.0,small_appliances_home_oven_and_coffee
6599,32bad49615b7c5da42609b98df97482c,portateis_casa_forno_e_cafe,51.0,81.0,1.0,50.0,18.0,8.0,13.0,small_appliances_home_oven_and_coffee
9110,4608a449f45011c89095f36265e75ec3,portateis_casa_forno_e_cafe,58.0,1257.0,1.0,450.0,16.0,20.0,16.0,small_appliances_home_oven_and_coffee
10544,514171f9d45a3d93574fc587ef775579,portateis_casa_forno_e_cafe,57.0,342.0,1.0,150.0,27.0,13.0,17.0,small_appliances_home_oven_and_coffee


In [54]:
products_copy.isnull().sum()

product_id                         0
product_category_name            610
product_name_lenght              610
product_description_lenght       610
product_photos_qty               610
product_weight_g                   2
product_length_cm                  2
product_height_cm                  2
product_width_cm                   2
product_category_name_english    610
dtype: int64

In [55]:
products_copy["product_category_name"] = (
    products_copy["product_category_name"].fillna("None")
)

In [56]:
products_copy['product_category_name'] = products_copy['product_category_name_english']

In [57]:
products_copy.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,00066f42aeeb9f3007548bb9d3f33c38,perfumery,53.0,596.0,6.0,300.0,20.0,16.0,16.0,perfumery
1,00088930e925c41fd95ebfe695fd2655,auto,56.0,752.0,4.0,1225.0,55.0,10.0,26.0,auto
2,0009406fd7479715e4bef61dd91f2462,bed_bath_table,50.0,266.0,2.0,300.0,45.0,15.0,35.0,bed_bath_table
3,000b8f95fcb9e0096488278317764d19,housewares,25.0,364.0,3.0,550.0,19.0,24.0,12.0,housewares
4,000d9be29b5207b54e86aa1b1ac54872,watches_gifts,48.0,613.0,4.0,250.0,22.0,11.0,15.0,watches_gifts


In [58]:
products_copy.columns

Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'product_category_name_english'],
      dtype='str')

In [59]:
products_copy

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,00066f42aeeb9f3007548bb9d3f33c38,perfumery,53.0,596.0,6.0,300.0,20.0,16.0,16.0,perfumery
1,00088930e925c41fd95ebfe695fd2655,auto,56.0,752.0,4.0,1225.0,55.0,10.0,26.0,auto
2,0009406fd7479715e4bef61dd91f2462,bed_bath_table,50.0,266.0,2.0,300.0,45.0,15.0,35.0,bed_bath_table
3,000b8f95fcb9e0096488278317764d19,housewares,25.0,364.0,3.0,550.0,19.0,24.0,12.0,housewares
4,000d9be29b5207b54e86aa1b1ac54872,watches_gifts,48.0,613.0,4.0,250.0,22.0,11.0,15.0,watches_gifts
...,...,...,...,...,...,...,...,...,...,...
32946,fff6177642830a9a94a0f2cba5e476d1,cool_stuff,54.0,842.0,1.0,1400.0,42.0,25.0,15.0,cool_stuff
32947,fff81cc3158d2725c0655ab9ba0f712c,health_beauty,54.0,643.0,1.0,58.0,16.0,16.0,11.0,health_beauty
32948,fff9553ac224cec9d15d49f5a263411f,fashion_bags_accessories,22.0,328.0,3.0,150.0,16.0,13.0,11.0,fashion_bags_accessories
32949,fffdb2d0ec8d6a61f0a0a0db3f25b441,computers_accessories,47.0,501.0,5.0,150.0,16.0,4.0,13.0,computers_accessories


In [60]:
products_copy.isnull().sum()

product_id                         0
product_category_name            610
product_name_lenght              610
product_description_lenght       610
product_photos_qty               610
product_weight_g                   2
product_length_cm                  2
product_height_cm                  2
product_width_cm                   2
product_category_name_english    610
dtype: int64

In [61]:
# Drop redundant translation column, but preserve physical weight and dimensions for logistics analysis
drop_cols = ["product_category_name_english"]
products_copy.drop(columns=[c for c in drop_cols if c in products_copy.columns], inplace=True)

# Fill any missing dimensional values with column medians
for dim_col in ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
    if dim_col in products_copy.columns:
        products_copy[dim_col] = products_copy[dim_col].fillna(products_copy[dim_col].median())

products_copy.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32951 non-null  float64
 6   product_length_cm           32951 non-null  float64
 7   product_height_cm           32951 non-null  float64
 8   product_width_cm            32951 non-null  float64
dtypes: float64(7), str(2)
memory usage: 3.7 MB


In [62]:
median_photos = products_copy["product_photos_qty"].median()
products_copy["product_photos_qty"] = products_copy["product_photos_qty"].fillna(
    median_photos
)

In [63]:
products_copy

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,00066f42aeeb9f3007548bb9d3f33c38,perfumery,53.0,596.0,6.0,300.0,20.0,16.0,16.0
1,00088930e925c41fd95ebfe695fd2655,auto,56.0,752.0,4.0,1225.0,55.0,10.0,26.0
2,0009406fd7479715e4bef61dd91f2462,bed_bath_table,50.0,266.0,2.0,300.0,45.0,15.0,35.0
3,000b8f95fcb9e0096488278317764d19,housewares,25.0,364.0,3.0,550.0,19.0,24.0,12.0
4,000d9be29b5207b54e86aa1b1ac54872,watches_gifts,48.0,613.0,4.0,250.0,22.0,11.0,15.0
...,...,...,...,...,...,...,...,...,...
32946,fff6177642830a9a94a0f2cba5e476d1,cool_stuff,54.0,842.0,1.0,1400.0,42.0,25.0,15.0
32947,fff81cc3158d2725c0655ab9ba0f712c,health_beauty,54.0,643.0,1.0,58.0,16.0,16.0,11.0
32948,fff9553ac224cec9d15d49f5a263411f,fashion_bags_accessories,22.0,328.0,3.0,150.0,16.0,13.0,11.0
32949,fffdb2d0ec8d6a61f0a0a0db3f25b441,computers_accessories,47.0,501.0,5.0,150.0,16.0,4.0,13.0


### Dataset Inspection: `olist_sellers`
**Schema Definition:**
- `seller_id`: Unique identifier for each merchant.
- `seller_zip_code_prefix`: 5-digit postal code prefix.
- `seller_city` & `seller_state`: Merchant location.

**Key Observations:**
- Contains 3,095 sellers with zero null values and zero duplicate seller IDs.
- We apply `clean_city` to standardize `seller_city` to match customer and geolocation naming conventions.

In [64]:
sellers = load_table("olist_sellers", chunk_size=50_000)
sellers_copy = sellers.copy()

In [65]:
sellers_copy.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,0015a82c2db000af6aaaf3ae2ecb0532,09080,santo andre,SP
1,001cca7ae9ae17fb1caed9dfb1094831,29156,cariacica,ES
2,001e6ad469a905060d959994f1b41e4f,24754,sao goncalo,RJ
3,002100f778ceb8431b7a1020ff7ab48f,14405,franca,SP
4,003554e2dce176b5555353e4f3555ac8,74565,goiania,GO


In [66]:
sellers_copy['seller_city'] = clean_city(sellers_copy['seller_city'])

In [67]:
sellers_copy.duplicated().sum()

np.int64(0)

In [68]:
sellers_copy.isnull().sum().sum()

np.int64(0)

### Dataset Inspection: `olist_orders`
**Schema Definition:**
- `order_id`: Unique order identifier.
- `customer_id`: Transaction-level foreign key referencing `olist_customers`.
- `order_status`: Order lifecycle stage (`delivered`, `shipped`, `canceled`, `invoiced`, `processing`, `unavailable`, `created`, `approved`).
- Timestamps: Purchase, payment approval, carrier delivery, customer receipt, and estimated delivery dates.

**Data Quality & Date Validation:**
- A common pitfall is naively filtering rows with `(delivery_date - purchase_date) >= 0`. That accidentally deletes all in-flight and canceled orders (~4,359 orders) because uncompleted orders legitimately have null delivery dates!
- Instead, we only validate timestamp logic for completed (`delivered`) orders where the delivery date is populated.
- We drop 6 corrupted records where orders were marked as `canceled` yet simultaneously recorded a customer delivery timestamp.

In [69]:
orders = load_table("olist_orders", chunk_size=50_000)

In [70]:
orders_copy = orders.copy()

In [71]:
orders_copy.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17


In [72]:
orders_copy['order_status'].unique()

<ArrowStringArray>
[  'delivered', 'unavailable',     'shipped',    'canceled',    'invoiced',
  'processing',    'approved',     'created']
Length: 8, dtype: str

In [73]:
orders_copy.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 13.0 MB


In [74]:
orders_copy.duplicated().sum()

np.int64(0)

In [75]:
orders_copy.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [76]:
orders_copy.shape

(99441, 8)

In [77]:
# 1. Identify corrupted delivery records (delivered before purchase)
corrupted_delivery_mask = (
    (orders_copy['order_status'] == 'delivered') &
    orders_copy['order_delivered_customer_date'].notna() &
    (orders_copy['order_delivered_customer_date'] < orders_copy['order_purchase_timestamp'])
)



In [78]:
# 2. Filter out only corrupted records, preserving canceled and in-flight orders
orders_copy = orders_copy[~corrupted_delivery_mask]

print(f"Cleaned orders count: {len(orders_copy):,} (preserves canceled and in-transit orders)")


Cleaned orders count: 99,441 (preserves canceled and in-transit orders)


In [79]:
orders_copy[(orders_copy['order_status']=='canceled') & (orders_copy['order_delivered_customer_date'].notna())]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
9837,1950d777989f6a877539f53795b4c3c3,1bccb206de9f0f25adc6871a1bcf77b2,canceled,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09
17037,2c45c33d2f9cb8ff8b1c86cc28c11c30,de4caa97afa80c8eeac2ff4c8da5b72e,canceled,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-10-14 10:40:50,2016-11-09 14:53:50,2016-12-08
39504,65d1e226dfaeb8cdc42f665422522d14,70fc57eeae292675927697fe03ad3ff5,canceled,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25
46160,770d331c84e5b214bd9dc70a10b829d0,6c57e6119369185e575b36712766b0ef,canceled,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-11 15:07:11,2016-10-14 15:07:11,2016-11-29
54136,8beb59392e21af5eb9547ae1a9938d06,bf609b5741f71697f65ce3852c5d2623,canceled,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-14 22:45:26,2016-10-19 18:47:43,2016-11-30
85048,dabf2b0e35b423f94618bf965fcb7514,5cdec0bb8cbdf53ffc8fdc212cd247c6,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30


In [80]:
invalid_orders_index = orders_copy[
    (orders_copy['order_status'] == 'canceled') & (orders_copy['order_delivered_customer_date'].notna())
].index

In [81]:
orders_copy = orders_copy.drop(invalid_orders_index)

In [82]:
orders_copy.shape

(99435, 8)

### Dataset Inspection: `olist_order_payments`
**Schema Definition:**
- `order_id`: Order foreign key.
- `payment_sequential`: Sequence index (allows splitting an order across multiple vouchers or cards).
- `payment_type`: Method used (`credit_card`, `boleto`, `voucher`, `debit_card`).
- `payment_installments`: Number of monthly installments.
- `payment_value`: Total monetary amount paid in this transaction.

**Key Observations:**
- A single order can have multiple payment methods (e.g., paying with a gift voucher and paying the remainder by credit card).
- When preparing an analytical order table, we group payment values by `order_id` and `payment_type` to preserve distinct payment methods per order without duplicate sequence noise.

In [83]:
order_payments = load_table("olist_order_payments", chunk_size=50_000)
orders_payments_copy = order_payments.copy()

In [84]:
orders_payments_copy.duplicated().sum()

np.int64(0)

In [85]:
orders_payments_copy.duplicated(subset=['order_id']).sum()

np.int64(4446)

In [86]:
orders_payments_copy.sort_values(by=[ 'payment_sequential'],ascending=False )

,payment_id,order_id,payment_sequential,payment_type,payment_installments,payment_value
39108,39109,fa65dad1b0e818e3ccc5cb0e39231352,29,voucher,1,19.26
39111,39112,fa65dad1b0e818e3ccc5cb0e39231352,28,voucher,1,29.05
4885,4886,fa65dad1b0e818e3ccc5cb0e39231352,27,voucher,1,66.02
79587,79588,fa65dad1b0e818e3ccc5cb0e39231352,26,voucher,1,28.27
32393,32394,ccf804e764ed5650cd8759557269dc13,26,voucher,1,23.10
...,...,...,...,...,...,...
103865,103866,f593228a7a25e9633ff78a4ce0d0d4cc,1,credit_card,8,165.80
103866,103867,387c2c33e495538e94878a500bf8e6d5,1,credit_card,1,87.58
103867,103868,d5639d8a973cec351be75a3186835068,1,boleto,1,160.89
103868,103869,0020262c8a370bd5a174ea6a2a267321,1,credit_card,2,100.55


In [87]:
orders_payments_copy[orders_payments_copy['order_id']=='fa65dad1b0e818e3ccc5cb0e39231352']

,payment_id,order_id,payment_sequential,payment_type,payment_installments,payment_value
4885,4886,fa65dad1b0e818e3ccc5cb0e39231352,27,voucher,1,66.02
9985,9986,fa65dad1b0e818e3ccc5cb0e39231352,4,voucher,1,29.16
14321,14322,fa65dad1b0e818e3ccc5cb0e39231352,1,voucher,1,3.71
17274,17275,fa65dad1b0e818e3ccc5cb0e39231352,9,voucher,1,1.08
19565,19566,fa65dad1b0e818e3ccc5cb0e39231352,10,voucher,1,12.86
23074,23075,fa65dad1b0e818e3ccc5cb0e39231352,2,voucher,1,8.51
24879,24880,fa65dad1b0e818e3ccc5cb0e39231352,25,voucher,1,3.68
28330,28331,fa65dad1b0e818e3ccc5cb0e39231352,5,voucher,1,0.66
29648,29649,fa65dad1b0e818e3ccc5cb0e39231352,6,voucher,1,5.02
32519,32520,fa65dad1b0e818e3ccc5cb0e39231352,11,voucher,1,4.03


In [88]:
orders_payments_copy[orders_payments_copy['payment_sequential']==1]

,payment_id,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,1,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,2,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,3,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,4,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,5,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
...,...,...,...,...,...,...
103881,103882,0406037ad97740d563a178ecc7a2075c,1,boleto,1,363.31
103882,103883,7b905861d7c825891d6347454ea7863f,1,credit_card,2,96.80
103883,103884,32609bbb3dd69b3c066a6860554a77bf,1,credit_card,1,47.77
103884,103885,b8b61059626efa996a60be9bb9320e10,1,credit_card,5,369.54


In [89]:
orders_payments_copy.drop(columns=['payment_sequential','payment_installments'], inplace=True)

In [90]:
orders_payments_copy.head()

,payment_id,order_id,payment_type,payment_value
0,1,b81ef226f3fe1789b1e8b2acac839d17,credit_card,99.33
1,2,a9810da82917af2d9aefd1278f1dcfa0,credit_card,24.39
2,3,25e8ea4e93396b6fa0d3dd708e76c1bd,credit_card,65.71
3,4,ba78997921bbcdc1373bb41e913ab953,credit_card,107.78
4,5,42fdf880ba16b47b59251dd489d4441a,credit_card,128.45


In [91]:
orders_payments_copy = orders_payments_copy.groupby(['order_id','payment_type']).agg({'payment_value':'sum'}).reset_index()

In [92]:
orders_payments_copy[orders_payments_copy['order_id']=='fa65dad1b0e818e3ccc5cb0e39231352']

,order_id,payment_type,payment_value
99453,fa65dad1b0e818e3ccc5cb0e39231352,voucher,457.99


In [93]:
print(orders_payments_copy['order_id'].nunique())
print(orders_copy['order_id'].nunique())

99440
99435


In [94]:
len(orders_copy[orders_copy['order_status'] == 'canceled'])

619

In [95]:
orders_copy[~orders_copy['order_id'].isin(orders_payments_copy['order_id'])].shape[0]

1

In [96]:
orders_copy[~orders_copy['order_id'].isin(orders_payments_copy['order_id'])].shape[0]

1

In [97]:
orders_copy = pd.merge(orders_copy, orders_payments_copy, how='inner', on='order_id')

### Dataset Inspection: `olist_order_items`
**Schema Definition:**
- `order_id`: Order foreign key.
- `order_item_id`: Sequential line item number within the order (Item 1, Item 2...).
- `product_id`: Product foreign key.
- `seller_id`: Merchant foreign key.
- `shipping_limit_date`: Seller shipping deadline.
- `price`: Item unit price.
- `freight_value`: Shipping and freight charge.

**Key Observations & Primary Key Integrity:**
- `order_item_id` must NEVER be dropped. When a customer buys multiple units of the same product, each unit shares the same `order_id` and `product_id`. Dropping `order_item_id` causes multi-item purchases to appear as false duplicates.
- The composite key `(order_id, order_item_id)` uniquely identifies each line item (112,650 items).

In [98]:
orders_items = load_table("olist_order_items", chunk_size=50_000)
orders_items_copy = orders_items.copy()

In [99]:
orders_items_copy.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [100]:
print(orders_copy['order_id'].nunique())
print(orders_items_copy['order_id'].nunique())


99434
98666


In [101]:
orders_copy = pd.merge(orders_copy, orders_items_copy, how='inner', on='order_id')

In [102]:
print(orders_items_copy['order_id'].nunique())


98666


In [103]:
orders_copy.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'payment_type',
 'payment_value',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value']

In [104]:
orders_copy.shape

(115128, 16)

In [105]:
orders_copy.info()

<class 'pandas.DataFrame'>
RangeIndex: 115128 entries, 0 to 115127
Data columns (total 16 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       115128 non-null  str           
 1   customer_id                    115128 non-null  str           
 2   order_status                   115128 non-null  str           
 3   order_purchase_timestamp       115128 non-null  datetime64[us]
 4   order_approved_at              115113 non-null  datetime64[us]
 5   order_delivered_carrier_date   113907 non-null  datetime64[us]
 6   order_delivered_customer_date  112628 non-null  datetime64[us]
 7   order_estimated_delivery_date  115128 non-null  datetime64[us]
 8   payment_type                   115128 non-null  str           
 9   payment_value                  115128 non-null  float64       
 10  order_item_id                  115128 non-null  int64         
 11  product_id 

In [106]:
orders_copy = orders_copy[(orders_copy['shipping_limit_date'] > orders_copy['order_purchase_timestamp'] ) &(orders_copy['shipping_limit_date']> orders_copy['order_approved_at'])]

In [107]:
orders_copy.shape

(114984, 16)

In [108]:
orders_copy.shape

(114984, 16)

In [109]:
orders_copy[orders_copy['order_id']=="8272b63d03f5f79c56e9e4120aec44ef"]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,payment_type,payment_value,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
58597,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,credit_card,196.11,1,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
58598,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,credit_card,196.11,2,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
58599,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,credit_card,196.11,3,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
58600,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,credit_card,196.11,4,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
58601,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,credit_card,196.11,5,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
58602,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,credit_card,196.11,6,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
58603,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,credit_card,196.11,7,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
58604,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,credit_card,196.11,8,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
58605,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,credit_card,196.11,9,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
58606,8272b63d03f5f79c56e9e4120aec44ef,fc3d1daec319d62d49bfb5e1f83123e9,delivered,2017-07-16 18:19:25,2017-07-17 18:25:23,2017-07-20 15:45:53,2017-07-31 18:03:02,2017-07-28,credit_card,196.11,10,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89


In [110]:
customer_copy.columns

Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='str')

In [111]:
pd.merge(orders_copy, customer_copy, how='inner', left_on='customer_id', right_on='customer_unique_id')

,order_id,customer_id_x,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,payment_type,payment_value,...,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id_y,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state


In [112]:
orders_copy = pd.merge(orders_copy, customer_copy, how='inner', on='customer_id')

In [113]:
orders_copy.shape

(114984, 20)

### Dataset Inspection: `olist_order_reviews`
**Schema Definition:**
- `review_id`: Unique review identifier.
- `order_id`: Order foreign key.
- `review_score`: Rating scale (1 to 5).
- `review_comment_title` & `review_comment_message`: Text feedback.
- `review_creation_date` & `review_answer_timestamp`: Review survey timestamps.

**Key Observations:**
- 551 orders have multiple review entries (e.g., a customer revised their feedback).
- Before joining reviews to orders, we deduplicate by `order_id` keeping the latest response (`review_answer_timestamp`) so line items are not artificially multiplied.
- We use a `left` join so orders without review submissions are fully preserved.

In [114]:
orders_reviews = load_table("olist_order_reviews", chunk_size=50_000)
orders_reviews_copy = orders_reviews.copy()

In [115]:
orders_reviews_copy.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,0001239bc1de2e33cb583967c2ca4c67,fc046d7776171871436844218f817d7d,5,NaN,NaN,2018-03-20,2018-03-20 18:36:04
1,0001cc6860aeaf5b9017fe4131a52e62,d4665434b01caa9dc3e3e78b3eb3593e,5,NaN,NaN,2018-06-22,2018-06-26 13:51:29
2,00020c7512a52e92212f12d3e37513c0,e28abf2eb2f1fbcbdc2dd0cd9a561671,5,Entrega rápida!,A entrega foi super rápida e o pendente é lind...,2018-04-25,2018-04-26 14:55:36
3,00032b0141443497c898b3093690af51,04fb47576993a3cb0c12d4b25eab6e4e,5,NaN,NaN,2017-05-30,2017-06-01 23:28:55
4,00034d88989f9a4c393bdcaec301537f,5f358d797a49fe2f24352f73426215f6,5,NaN,NaN,2017-08-12,2017-08-13 19:56:53


In [116]:
orders_reviews_copy.shape

(99224, 7)

In [117]:
# since my order_copy is clean so i check which order_id is present in order_reviews_copy but not present in orders_copy
orders_reviews_copy[orders_reviews_copy['order_id'].isin(orders_copy['order_id'])].shape

(98330, 7)

In [118]:
# Deduplicate reviews: keep latest review per order_id to prevent row multiplication
orders_reviews_clean = (
    orders_reviews_copy.sort_values("review_answer_timestamp")
    .drop_duplicates(subset=["order_id"], keep="last")
)
print(f"Total reviews: {len(orders_reviews_copy):,} | Unique orders reviewed: {len(orders_reviews_clean):,}")

Total reviews: 99,224 | Unique orders reviewed: 98,673


In [119]:
orders_copy = pd.merge(orders_copy, orders_reviews_clean, how="left", on="order_id")
print("Final merged dataset shape:", orders_copy.shape)
orders_copy.head(3)

Final merged dataset shape: (114984, 26)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,payment_type,payment_value,...,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,credit_card,72.19,...,871766c5855e863f6eccc05f988b23cb,28013,Campos Dos Goytacazes,RJ,97ca439bc427b48bc1cd7177abe71365,5.0,NaN,"Perfeito, produto entregue antes do combinado.",2017-09-21,2017-09-22 10:57:03
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,credit_card,259.83,...,eb28e67c4c0b83846050ddfb8a35d051,15775,Santa Fe Do Sul,SP,7b07bacd811c4117b742569b04ce3580,4.0,NaN,NaN,2017-05-13,2017-05-15 11:34:13
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,credit_card,216.87,...,3818d81c6709e39d06b2738a8d3a2474,35661,Para De Minas,MG,0c5b33dea94867d1ac402749e5438e8b,5.0,NaN,Chegou antes do prazo previsto e o produto sur...,2018-01-23,2018-01-23 16:06:31


### 8. Production Export: Parquet Storage & Star Schema Artifacts
We export the cleaned data into `data/processed/` using the columnar **Parquet** format:
- **High Performance**: Columnar compression is 4-5x smaller than CSV and loads significantly faster.
- **Type Preservation**: Datetime, float, and integer types remain strictly preserved without CSV re-parsing errors.
- **Star Schema Architecture**: We save dedicated dimension and fact tables for clean relational modeling in Power BI and SQL analytical warehouses.


In [120]:
from e_commerce_sales_analysis.config import DATA_PROCESSED_DIR

# Ensure processed data directory exists
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 1. Export Master Merged Table for Quick EDA & Modeling
orders_copy.to_parquet(DATA_PROCESSED_DIR / "olist_master_cleaned.parquet", index=False)
print("Exported: olist_master_cleaned.parquet", orders_copy.shape)

# 2. Export Star Schema Dimensional Model for Power BI & SQL
customer_copy.to_parquet(DATA_PROCESSED_DIR / "dim_customers.parquet", index=False)
products_copy.to_parquet(DATA_PROCESSED_DIR / "dim_products.parquet", index=False)
sellers_copy.to_parquet(DATA_PROCESSED_DIR / "dim_sellers.parquet", index=False)
orders_copy[['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']].drop_duplicates().to_parquet(DATA_PROCESSED_DIR / "dim_orders.parquet", index=False)
orders_items_copy.to_parquet(DATA_PROCESSED_DIR / "fact_order_items.parquet", index=False)
orders_payments_copy.to_parquet(DATA_PROCESSED_DIR / "fact_order_payments.parquet", index=False)
orders_reviews_clean.to_parquet(DATA_PROCESSED_DIR / "dim_order_reviews.parquet", index=False)

print("All Star Schema tables successfully exported to:", DATA_PROCESSED_DIR)


Exported: olist_master_cleaned.parquet (114984, 26)
All Star Schema tables successfully exported to: D:\E-commerce Sales Analysis\data\processed
